In [ ]:
import sys, json, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import logit
from sklearn.model_selection import (StratifiedShuffleSplit, StratifiedKFold, GridSearchCV,
                                     RandomizedSearchCV, cross_val_predict)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss, roc_curve,
                             precision_recall_curve, confusion_matrix, f1_score, log_loss)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
import joblib
import sklearn

warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import features as F

SEED = 42
rng = np.random.default_rng(SEED)
DATA = ROOT / "data" / "findex_microdata_2025_labelled_update112425.csv"
FIG, RES, MOD = ROOT / "figures", ROOT / "results", ROOT / "models"
for p in (FIG, RES, MOD):
    p.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| seed", SEED)

In [ ]:
# Load the data
raw = pd.read_csv(DATA, low_memory=False)
sha = hashlib.sha256(DATA.read_bytes()).hexdigest()
print(raw.shape, "| SHA-256:", sha[:16], "…")
print("Economies:", raw.economy.nunique(), "| fieldwork year:", raw.year.unique())
manifest = {"file": DATA.name, "rows": int(raw.shape[0]), "cols": int(raw.shape[1]), "sha256": sha,
            "source": "World Bank Microdata Library, catalogue 7860, WLD_2024_FINDEX_v02_M"}
json.dump(manifest, open(RES / "data_manifest.json", "w"), indent=2)

In [ ]:
#2. Data cleaning
pop = F.analysis_population(raw)
flow = [("All respondents", len(raw), raw.economy.nunique()),
        ("Economies with resilience module", len(pop), pop.economy.nunique())]
pop["FRAGILE"] = F.make_fragility_target(pop)
pop["MEDBORROW"] = F.make_medical_borrowing_target(pop)
df = pop[pop["FRAGILE"].notna()].copy()
flow.append(("Valid FRAGILE label (DK/refused removed)", len(df), df.economy.nunique()))
flow = pd.DataFrame(flow, columns=["step", "adults", "economies"])
flow.to_csv(RES / "participant_flow.csv", index=False)
flow


In [ ]:
X_all = F.build_features(df)
y = df["FRAGILE"].astype(int).values
miss = X_all.isna().mean().sort_values(ascending=False).rename("share_missing").to_frame()
miss.to_csv(RES / "missingness.csv")
print(f"FRAGILE prevalence: {y.mean():.3f}  (n = {len(y):,}; positives = {y.sum():,})")
miss.head(10)

In [ ]:
#3. Exploratory analysis
lab_inc = {1: "Q1 poorest", 2: "Q2", 3: "Q3", 4: "Q4", 5: "Q5 richest"}
eda = X_all.assign(FRAGILE=y)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})
reg = eda.groupby("region", observed=True)["FRAGILE"].agg(["mean", "size"]).sort_values("mean")
axes[0].barh(reg.index.astype(str), reg["mean"] * 100, color="#3b6ea5")
for i, (m, n) in enumerate(zip(reg["mean"], reg["size"])):
    axes[0].text(m * 100 + 0.8, i, f"{m*100:.1f}%  (n={n:,})", va="center", fontsize=9)
axes[0].set_xlabel("Financially fragile (% of adults, unweighted)")
axes[0].set_title("A. Financial fragility by World Bank region")
axes[0].set_xlim(0, 80)
inc = eda.groupby("inc_q")["FRAGILE"].mean().rename(index=lab_inc)
axes[1].bar(inc.index, inc.values * 100, color=sns.color_palette("crest", 5))
for i, v in enumerate(inc.values):
    axes[1].text(i, v * 100 + 1, f"{v*100:.1f}%", ha="center", fontsize=9)
axes[1].set_ylabel("Financially fragile (%)")
axes[1].set_title("B. Fragility by within-economy income quintile")
axes[1].set_ylim(0, 75)
fig.suptitle("Figure 1. Prevalence of financial fragility, Global Findex 2025 (98 economies)", y=1.02)
fig.savefig(FIG / "fig1_prevalence.png"); plt.show()

In [ ]:
tab1_vars = ["female", "rural", "in_workforce", "account_fin", "saved_any", "borrowed_medical",
             "food_on_credit", "insurance_pay", "gov_support", "disaster_income_loss", "internet_use"]
table1 = (eda.groupby("FRAGILE")[tab1_vars + ["age"]].mean().T
          .rename(columns={0: "Not fragile", 1: "Fragile"}))
table1["Difference"] = table1["Fragile"] - table1["Not fragile"]
table1.loc["n"] = [int((y == 0).sum()), int((y == 1).sum()), np.nan]
table1.round(3).to_csv(RES / "table1_characteristics.csv")
table1.round(3)

In [ ]:
#4. Locked test set
strat = df["economy"].astype(str) + "_" + df["FRAGILE"].astype(int).astype(str)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
dev_idx, test_idx = next(sss.split(df, strat))
split = pd.Series("dev", index=df.index); split.iloc[test_idx] = "test"
split.rename("split").to_frame().assign(wpid=df["wpid_random"].values).to_csv(RES / "split_assignment.csv")
FEATS = F.FEATURES_FRAGILE
F.assert_no_leakage(FEATS)
X, Xd, Xt = X_all[FEATS], X_all[FEATS].iloc[dev_idx], X_all[FEATS].iloc[test_idx]
yd, yt = y[dev_idx], y[test_idx]
wt = df["wgt"].values[test_idx]
econ_t = df["economy"].values[test_idx]
print(f"Development: {len(yd):,} ({yd.sum():,} positives) | Locked test: {len(yt):,} ({yt.sum():,} positives)")

In [ ]:
#5. Feature engineering and leakage-safe pipelines
def linear_preprocessor(feats):
    num = [c for c in ["age"] if c in feats]
    ordc = [c for c in F.ORDINAL if c in feats]
    cat = [c for c in F.CATEGORICAL if c in feats]
    binr = [c for c in feats if c not in num + ordc + cat]
    steps = []
    if num:
        steps.append(("age", make_pipeline(SimpleImputer(strategy="median"),
                                           SplineTransformer(n_knots=4, degree=3), StandardScaler()), num))
    if ordc:
        steps.append(("ord", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                           OneHotEncoder(drop="first", handle_unknown="ignore")), ordc))
    if cat:
        steps.append(("cat", make_pipeline(FunctionTransformer(F.to_object, feature_names_out="one-to-one"),
                                           OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)), cat))
    if binr:
        steps.append(("bin", SimpleImputer(strategy="most_frequent", add_indicator=True), binr))
    return ColumnTransformer(steps, verbose_feature_names_out=True)

def make_lr(feats, C=1.0):
    return Pipeline([("prep", linear_preprocessor(feats)),
                     ("clf", LogisticRegression(C=C, max_iter=3000))])

def make_hgb(**kw):
    return HistGradientBoostingClassifier(categorical_features="from_dtype", early_stopping=True,
                                          validation_fraction=0.1, n_iter_no_change=20,
                                          max_iter=600, random_state=SEED, **kw)

inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [ ]:
#6. Model training
t0 = time.time()
b0 = DummyClassifier(strategy="prior").fit(Xd, yd)
b1 = make_lr(F.BASELINE_FEATURES).fit(Xd[F.BASELINE_FEATURES], yd)

lr_search = GridSearchCV(make_lr(FEATS), {"clf__C": [0.001, 0.01, 0.1, 1.0, 10.0]},
                         scoring="neg_log_loss", cv=inner, n_jobs=2)
lr_search.fit(Xd, yd)
m1 = lr_search.best_estimator_
print("M1 best C:", lr_search.best_params_, f"| CV log-loss {-lr_search.best_score_:.4f}")

# Ranges widened once (logged): a first run selected learning_rate=0.03 and max_leaf_nodes=63,
# both on the edge of the original grid (Stage 11 Phase 7 completion check).
hgb_space = {"learning_rate": [0.015, 0.03, 0.05, 0.08, 0.12],
             "max_leaf_nodes": [15, 31, 63, 127],
             "min_samples_leaf": [20, 50, 100, 200],
             "l2_regularization": [0.0, 0.1, 1.0, 5.0],
             "max_features": [0.5, 0.8, 1.0]}
hgb_search = RandomizedSearchCV(make_hgb(), hgb_space, n_iter=20, scoring="neg_log_loss",
                                cv=inner, random_state=SEED, n_jobs=2)
hgb_search.fit(Xd, yd)
m2 = hgb_search.best_estimator_
print("M2 best params:", hgb_search.best_params_, f"| CV log-loss {-hgb_search.best_score_:.4f}")
print(f"Training time: {time.time()-t0:.0f} s")
pd.DataFrame(hgb_search.cv_results_).sort_values("rank_test_score")[
    ["params", "mean_test_score", "std_test_score"]].head(5)

In [ ]:
#6.1 Out-of-fold predictions on the development set
oof = {
    "B1 age-sex-income": cross_val_predict(make_lr(F.BASELINE_FEATURES), Xd[F.BASELINE_FEATURES], yd,
                                           cv=inner, method="predict_proba", n_jobs=2)[:, 1],
    "M1 penalised logistic": cross_val_predict(make_lr(FEATS, **{"C": lr_search.best_params_["clf__C"]}),
                                               Xd, yd, cv=inner, method="predict_proba", n_jobs=2)[:, 1],
    "M2 gradient boosting": cross_val_predict(make_hgb(**hgb_search.best_params_), Xd, yd, cv=inner,
                                              method="predict_proba", n_jobs=2)[:, 1],
}

def calib_int_slope(y_true, p):
    lp = logit(np.clip(p, 1e-6, 1 - 1e-6)).reshape(-1, 1)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp, y_true).coef_[0, 0]
    # calibration-in-the-large: intercept with slope fixed at 1 (offset model)
    from scipy.optimize import minimize_scalar
    off = lp.ravel()
    nll = lambda a: -np.sum(y_true * (a + off) - np.log1p(np.exp(a + off)))
    intercept = minimize_scalar(nll, bounds=(-3, 3), method="bounded").x
    return intercept, slope

def metric_row(y_true, p, w=None):
    ci, cs = calib_int_slope(y_true, p)
    brier = brier_score_loss(y_true, p)
    brier0 = brier_score_loss(y_true, np.full_like(p, y_true.mean()))
    return {"AUROC": roc_auc_score(y_true, p), "AUPRC": average_precision_score(y_true, p),
            "Brier": brier, "Scaled Brier": 1 - brier / brier0, "Log-loss": log_loss(y_true, p),
            "Calib. intercept": ci, "Calib. slope": cs}

oof_tab = pd.DataFrame({k: metric_row(yd, v) for k, v in oof.items()}).T
fold_auc = {k: [roc_auc_score(yd[va], v[va]) for _, va in inner.split(Xd, yd)] for k, v in oof.items()}
oof_tab["AUROC fold SD"] = [np.std(fold_auc[k]) for k in oof_tab.index]
oof_tab.round(4).to_csv(RES / "dev_oof_metrics.csv")
oof_tab.round(4)

In [ ]:
#6.2 Pre-set selection rule and screening threshold
a1, a2 = oof_tab.loc["M1 penalised logistic", "AUROC"], oof_tab.loc["M2 gradient boosting", "AUROC"]
selected = "M2 gradient boosting" if (a2 - a1) >= 0.02 else "M1 penalised logistic"
print(f"OOF AUROC M1={a1:.4f}, M2={a2:.4f}, Δ={a2-a1:+.4f} → selected (pre-set rule): {selected}")

def threshold_for_sensitivity(y_true, p, target=0.80):
    fpr, tpr, thr = roc_curve(y_true, p)
    return float(thr[np.argmax(tpr >= target)])

thresholds = {k: threshold_for_sensitivity(yd, v) for k, v in oof.items()}
thresholds

In [ ]:
# 7. Locked test-set evaluation
models = {"B0 prevalence": b0, "B1 age-sex-income": b1, "M1 penalised logistic": m1, "M2 gradient boosting": m2}
feat_sets = {"B0 prevalence": FEATS, "B1 age-sex-income": F.BASELINE_FEATURES,
             "M1 penalised logistic": FEATS, "M2 gradient boosting": FEATS}
pt = {k: m.predict_proba(Xt[feat_sets[k]])[:, 1] for k, m in models.items()}

econs = np.unique(econ_t)
idx_by_econ = {e: np.where(econ_t == e)[0] for e in econs}
B = 1000
boot_idx = []
for b in range(B):
    draw = rng.choice(econs, size=len(econs), replace=True)
    boot_idx.append(np.concatenate([idx_by_econ[e] for e in draw]))

def boot_ci(fn, *preds):
    vals = np.array([fn(yt[ix], *[p[ix] for p in preds]) for ix in boot_idx])
    return np.percentile(vals, [2.5, 97.5])

rows = []
for k, p in pt.items():
    r = metric_row(yt, p) if k != "B0 prevalence" else {
        "AUROC": 0.5, "AUPRC": yt.mean(), "Brier": brier_score_loss(yt, p), "Scaled Brier": 0.0,
        "Log-loss": log_loss(yt, p), "Calib. intercept": np.nan, "Calib. slope": np.nan}
    if k != "B0 prevalence":
        lo, hi = boot_ci(roc_auc_score, p); r["AUROC 95% CI"] = f"{lo:.3f}–{hi:.3f}"
        lo, hi = boot_ci(average_precision_score, p); r["AUPRC 95% CI"] = f"{lo:.3f}–{hi:.3f}"
        lo, hi = boot_ci(brier_score_loss, p); r["Brier 95% CI"] = f"{lo:.3f}–{hi:.3f}"
        r["AUROC (survey-weighted)"] = roc_auc_score(yt, p, sample_weight=wt)
    rows.append(pd.Series(r, name=k))
test_tab = pd.DataFrame(rows)
test_tab.to_csv(RES / "test_metrics.csv")
test_tab.round(4)

In [ ]:
def paired(fn, pa, pb):
    d = np.array([fn(yt[ix], pa[ix]) - fn(yt[ix], pb[ix]) for ix in boot_idx])
    return fn(yt, pa) - fn(yt, pb), *np.percentile(d, [2.5, 97.5])

comparisons = [("M1 penalised logistic", "B1 age-sex-income"), ("M2 gradient boosting", "B1 age-sex-income"),
               ("M2 gradient boosting", "M1 penalised logistic")]
diff_tab = pd.DataFrame([
    {"comparison": f"{a} − {b}", "ΔAUROC": paired(roc_auc_score, pt[a], pt[b])[0],
     "95% CI low": paired(roc_auc_score, pt[a], pt[b])[1], "95% CI high": paired(roc_auc_score, pt[a], pt[b])[2]}
    for a, b in comparisons])
diff_tab.to_csv(RES / "test_auc_differences.csv", index=False)
diff_tab.round(4)

In [ ]:
#7.1 Operating point at the pre-set threshold (sensitivity target 0.80 on OOF)
op_rows = []
for k in ["B1 age-sex-income", "M1 penalised logistic", "M2 gradient boosting"]:
    pred = (pt[k] >= thresholds[k]).astype(int)
    tn, fp, fn_, tp = confusion_matrix(yt, pred).ravel()
    op_rows.append({"model": k, "threshold": thresholds[k], "sensitivity": tp / (tp + fn_),
                    "specificity": tn / (tn + fp), "PPV": tp / (tp + fp), "NPV": tn / (tn + fn_),
                    "F1": f1_score(yt, pred), "share flagged": pred.mean(),
                    "TP": tp, "FP": fp, "FN": fn_, "TN": tn})
op_tab = pd.DataFrame(op_rows).set_index("model")
op_tab.to_csv(RES / "test_operating_point.csv")
op_tab.round(3)

In [ ]:
# 7.2 Figures: ROC, calibration and decision curves (locked test set)
pal = {"B1 age-sex-income": "#9e9e9e", "M1 penalised logistic": "#3b6ea5", "M2 gradient boosting": "#d9822b"}
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for k, c in pal.items():
    fpr, tpr, _ = roc_curve(yt, pt[k])
    axes[0].plot(fpr, tpr, color=c, lw=2, label=f"{k} (AUROC {test_tab.loc[k,'AUROC']:.3f})")
    prob_true, prob_pred = calibration_curve(yt, pt[k], n_bins=10, strategy="quantile")
    axes[1].plot(prob_pred, prob_true, "o-", color=c, lw=1.8, ms=4,
                 label=f"{k} (slope {test_tab.loc[k,'Calib. slope']:.2f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1); axes[0].set(xlabel="1 − specificity", ylabel="Sensitivity",
                                                       title="A. ROC curves")
axes[0].legend(loc="lower right", fontsize=8.5)
axes[1].plot([0, 1], [0, 1], "k--", lw=1); axes[1].set(xlabel="Mean predicted probability (decile)",
                                                       ylabel="Observed proportion fragile",
                                                       title="B. Calibration (deciles of risk)")
axes[1].legend(loc="upper left", fontsize=8.5)
ths = np.linspace(0.05, 0.80, 76)
prev = yt.mean()
def net_benefit(p, t):
    pred = p >= t; n = len(yt)
    tp = np.sum(pred & (yt == 1)); fp = np.sum(pred & (yt == 0))
    return tp / n - fp / n * t / (1 - t)
for k, c in pal.items():
    axes[2].plot(ths, [net_benefit(pt[k], t) for t in ths], color=c, lw=2, label=k)
axes[2].plot(ths, [prev - (1 - prev) * t / (1 - t) for t in ths], color="black", lw=1, ls=":", label="Target all")
axes[2].axhline(0, color="black", lw=1, label="Target none")
axes[2].set(ylim=(-0.05, prev + 0.03), xlabel="Threshold probability", ylabel="Net benefit",
            title="C. Decision curve")
axes[2].legend(fontsize=8.5)
fig.suptitle("Figure 2. Discrimination, calibration and net benefit on the locked test set "
             f"(n = {len(yt):,}; {yt.sum():,} fragile)", y=1.02)
fig.savefig(FIG / "fig2_roc_calibration_dca.png"); plt.show()

In [ ]:
# 8. Transfer test: leave-one-region-out
reg_d = Xd["region"].astype(str).values
tr_rows = []
for r in sorted(set(reg_d)):
    tr, te = reg_d != r, reg_d == r
    Xtr, Xte = Xd[tr].copy(), Xd[te].copy()
    for name, est in [("M1 penalised logistic", make_lr(FEATS, C=lr_search.best_params_["clf__C"])),
                      ("M2 gradient boosting", make_hgb(**hgb_search.best_params_))]:
        est.fit(Xtr, yd[tr]); p = est.predict_proba(Xte)[:, 1]
        ci, cs = calib_int_slope(yd[te], p)
        tr_rows.append({"held-out region": r, "model": name, "n": int(te.sum()), "positives": int(yd[te].sum()),
                        "observed %": yd[te].mean() * 100, "mean predicted %": p.mean() * 100,
                        "AUROC": roc_auc_score(yd[te], p), "Calib. intercept": ci, "Calib. slope": cs})
transfer = pd.DataFrame(tr_rows)
transfer.to_csv(RES / "transfer_leave_one_region_out.csv", index=False)
transfer_wide = transfer.pivot(index="held-out region", columns="model",
                               values=["AUROC", "Calib. slope", "mean predicted %"])
transfer_wide.columns = [f"{m.split()[0]} {v}" for v, m in transfer_wide.columns]
transfer_wide.insert(0, "observed %", transfer.groupby("held-out region")["observed %"].first())
transfer_wide.round(3)

In [ ]:
#9. Subgroup and fairness audit (locked test set, selected model and comparator)
sel_threshold = thresholds[selected]
te_frame = Xt.assign(y=yt, p_sel=pt[selected], p_m1=pt["M1 penalised logistic"], economy=econ_t)
groups = {
    "Sex": te_frame["female"].map({1.0: "Female", 0.0: "Male"}),
    "Age band": pd.cut(te_frame["age"], [14, 24, 34, 44, 54, 64, 120],
                       labels=["15–24", "25–34", "35–44", "45–54", "55–64", "65+"]),
    "Income quintile": te_frame["inc_q"].map(lab_inc),
    "Residence": te_frame["rural"].map({1.0: "Rural", 0.0: "Urban"}),
    "Education": te_frame["educ"].map({1.0: "Primary or less", 2.0: "Secondary", 3.0: "Tertiary"}),
    "Region": te_frame["region"].astype(str),
    "Focal country": pd.Series(np.where(te_frame["economy"] == "Bangladesh", "Bangladesh", None),
                               index=te_frame.index),
}
sg = []
for gname, g in groups.items():
    for lvl in g.dropna().unique():
        m = (g == lvl).values
        yy, pp = te_frame["y"].values[m], te_frame["p_sel"].values[m]
        pos = int(yy.sum()); flag = pp >= sel_threshold
        sg.append({"dimension": gname, "group": str(lvl), "n": int(m.sum()), "positives": pos,
                   "observed %": yy.mean() * 100, "mean predicted %": pp.mean() * 100,
                   "AUROC": roc_auc_score(yy, pp) if 0 < pos < m.sum() else np.nan,
                   "sensitivity": flag[yy == 1].mean() if pos else np.nan,
                   "share flagged %": flag.mean() * 100,
                   "status": "finding" if pos >= 30 else "descriptive (<30 positives)"})
subgroups = pd.DataFrame(sg).sort_values(["dimension", "group"])
subgroups.to_csv(RES / "subgroup_audit.csv", index=False)
subgroups.round(3)

In [ ]:
plot_sg = subgroups[subgroups["dimension"].isin(["Sex", "Income quintile", "Residence", "Education", "Region"])].copy()
plot_sg["label"] = plot_sg["dimension"] + ": " + plot_sg["group"]
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
axes[0].scatter(plot_sg["AUROC"], plot_sg["label"], color="#3b6ea5", zorder=3)
axes[0].axvline(test_tab.loc[selected, "AUROC"], color="black", ls="--", lw=1, label="Overall")
axes[0].set(xlabel="AUROC", title="A. Discrimination by subgroup"); axes[0].legend()
axes[1].scatter(plot_sg["observed %"], plot_sg["label"], color="black", marker="|", s=200, label="Observed %", zorder=3)
axes[1].scatter(plot_sg["mean predicted %"], plot_sg["label"], color="#d9822b", label="Mean predicted %", zorder=3)
axes[1].scatter(plot_sg["sensitivity"] * 100, plot_sg["label"], color="#3b6ea5", marker="s", label="Sensitivity %", zorder=3)
axes[1].set(xlabel="Percent", title="B. Calibration-in-the-large and sensitivity at threshold")
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=9)
fig.suptitle(f"Figure 3. Subgroup audit of the selected model ({selected}), locked test set", y=1.01)
fig.savefig(FIG / "fig3_subgroups.png"); plt.show()

In [ ]:
# 10. Interpretation
names = m1.named_steps["prep"].get_feature_names_out()
coefs = m1.named_steps["clf"].coef_[0]
or_tab = pd.DataFrame({"term": names, "coef": coefs, "odds_ratio": np.exp(coefs)})
or_tab = or_tab[~or_tab["term"].str.startswith("age__")]  # spline basis terms are not interpretable singly
or_tab["term"] = (or_tab["term"].str.replace(r"^(bin|cat|ord)__", "", regex=True)
                  .str.replace("missingindicator_", "missing: "))
or_tab["label"] = [F.pretty_term(t) for t in or_tab["term"]]
or_tab = or_tab.reindex(or_tab["coef"].abs().sort_values(ascending=False).index)
or_tab.round(3).to_csv(RES / "m1_odds_ratios.csv", index=False)
or_tab.head(20).round(3)

In [ ]:
sub = rng.choice(len(yt), size=min(8000, len(yt)), replace=False)
perm = {}
for k in ["M1 penalised logistic", "M2 gradient boosting"]:
    r = permutation_importance(models[k], Xt.iloc[sub], yt[sub], scoring="roc_auc", n_repeats=5,
                               random_state=SEED, n_jobs=2)
    perm[k] = pd.Series(r.importances_mean, index=FEATS)
perm_tab = pd.DataFrame(perm).sort_values("M2 gradient boosting", ascending=False)
perm_tab.to_csv(RES / "permutation_importance.csv")
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={"wspace": 0.85})
top = perm_tab.head(15)
top_lab = [F.LABELS.get(c, c) for c in top.index[::-1]]
axes[0].barh(top_lab, top["M2 gradient boosting"][::-1], color="#d9822b", label="M2 gradient boosting")
axes[0].scatter(top["M1 penalised logistic"][::-1], top_lab, color="#3b6ea5", zorder=3, label="M1 logistic")
axes[0].set(xlabel="Mean drop in test AUROC when permuted", title="A. Top-15 predictors (grouped permutation importance)")
axes[0].legend(loc="lower right", fontsize=9, framealpha=1)
top_or = or_tab[~or_tab["term"].str.startswith("missing")].head(15).iloc[::-1]
axes[1].scatter(top_or["odds_ratio"], [F.pretty_term(t) for t in top_or["term"]],
                color=np.where(top_or["odds_ratio"] > 1, "#b23a48", "#2e7d32"), zorder=3)
axes[1].axvline(1, color="black", lw=1); axes[1].set_xscale("log")
from matplotlib.ticker import FixedLocator, NullLocator, FormatStrFormatter
axes[1].xaxis.set_major_locator(FixedLocator([0.4, 0.5, 0.7, 1.0, 1.4, 2.0]))
axes[1].xaxis.set_minor_locator(NullLocator())
axes[1].xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
axes[1].set(xlabel="Adjusted odds ratio (log scale)", title="B. M1 odds ratios: 15 largest effects")
fig.suptitle("Figure 4. What the models rely on (associations, not causal effects)", y=1.01)
fig.savefig(FIG / "fig4_importance.png"); plt.show()

In [ ]:
# 11. Secondary model: borrowing for health or medical purposes
dm = pop[pop["MEDBORROW"].notna()].copy()
Xm_all = F.build_features(dm)[F.FEATURES_MEDBORROW]
F.assert_no_leakage(Xm_all.columns)
ym = dm["MEDBORROW"].astype(int).values
strat_m = dm["economy"].astype(str) + "_" + dm["MEDBORROW"].astype(int).astype(str)
d_i, t_i = next(StratifiedShuffleSplit(1, test_size=0.2, random_state=SEED).split(dm, strat_m))
print(f"MEDBORROW: n={len(ym):,}, prevalence={ym.mean():.3f}")
sec = {}
lr_m = GridSearchCV(make_lr(F.FEATURES_MEDBORROW), {"clf__C": [0.01, 0.1, 1.0]}, scoring="neg_log_loss",
                    cv=inner, n_jobs=2).fit(Xm_all.iloc[d_i], ym[d_i]).best_estimator_
hgb_m = make_hgb(**hgb_search.best_params_).fit(Xm_all.iloc[d_i], ym[d_i])
base_m = make_lr(F.BASELINE_FEATURES).fit(Xm_all.iloc[d_i][F.BASELINE_FEATURES], ym[d_i])
for k, est, cols in [("B1 age-sex-income", base_m, F.BASELINE_FEATURES),
                     ("M1 penalised logistic", lr_m, F.FEATURES_MEDBORROW),
                     ("M2 gradient boosting", hgb_m, F.FEATURES_MEDBORROW)]:
    sec[k] = metric_row(ym[t_i], est.predict_proba(Xm_all.iloc[t_i][cols])[:, 1])
sec_tab = pd.DataFrame(sec).T
sec_tab.to_csv(RES / "secondary_medborrow_test_metrics.csv")
r = permutation_importance(hgb_m, Xm_all.iloc[t_i], ym[t_i], scoring="roc_auc", n_repeats=5,
                           random_state=SEED, n_jobs=2)
sec_imp = pd.Series(r.importances_mean, index=F.FEATURES_MEDBORROW).sort_values(ascending=False)
sec_imp.to_csv(RES / "secondary_medborrow_importance.csv")
display_tab = sec_tab.round(4); display_tab

In [ ]:
# 12. Save deployable artefacts
final_params = hgb_search.best_params_ if selected.startswith("M2") else {"C": lr_search.best_params_["clf__C"]}
final = make_hgb(**final_params) if selected.startswith("M2") else make_lr(FEATS, **final_params)
final.fit(X, y)
bundle = {"model": final, "features": FEATS, "threshold": thresholds[selected], "selected": selected,
          "target": "FRAGILE (fin24 == 7 or fin24a == 1)", "sklearn": sklearn.__version__,
          "trained_rows": int(len(y)), "data_sha256": sha}
joblib.dump(bundle, MOD / "fragility_model.joblib")
joblib.dump({"model": m1, "features": FEATS, "threshold": thresholds["M1 penalised logistic"]},
            MOD / "fragility_logistic_dev.joblib")
summary = {
    "n_analysis": int(len(y)), "prevalence": float(y.mean()), "n_dev": int(len(yd)), "n_test": int(len(yt)),
    "economies": int(df.economy.nunique()), "selected": selected,
    "best_C": lr_search.best_params_["clf__C"], "hgb_params": hgb_search.best_params_,
    "thresholds": thresholds,
    "test": test_tab.drop(columns=[c for c in test_tab.columns if "CI" in c]).round(4).to_dict(orient="index"),
    "test_ci": test_tab[[c for c in test_tab.columns if "CI" in c]].to_dict(orient="index"),
    "auc_diffs": diff_tab.round(4).to_dict(orient="records"),
    "operating_point": op_tab.round(4).to_dict(orient="index"),
    "secondary": sec_tab.round(4).to_dict(orient="index"),
}
json.dump(summary, open(RES / "summary.json", "w"), indent=2, default=float)
print(json.dumps({k: summary[k] for k in ["n_analysis", "prevalence", "selected", "thresholds"]}, indent=2, default=float))

Findings and next steps. 
In 98,815 adults across 98 economies, 41.2% were financially fragile, rising from 26% in the richest to 59% in the poorest within-economy income quintile and peaking at 63% in South Asia. Both models clearly beat the age–sex–income baseline (test AUROC 0.646): penalised logistic regression reached 0.769 (95% CI 0.750–0.788) and gradient boosting 0.779 (0.760–0.797), with calibration slopes of about 1.0 for both. Boosting's gain (ΔAUROC 0.010, 95% CI 0.006–0.015) fell below the pre-set 0.02 minimal important difference, so the pre-registered rule selected the transparent logistic model, which reaches 80% sensitivity by flagging 57% of adults (PPV 0.58). Income, saving, region, education and financial worry carried most of the signal, and debt and shock markers (food bought on credit, OR 1.38; borrowing for medical costs, OR 1.36; income lost to a disaster, OR 1.39) raised the odds of fragility, while paying for insurance lowered them (OR 0.75). The ranking transferred to regions the model had not seen (AUROC 0.70–0.80) but absolute risk did not (South Asia: 50% predicted vs 63% observed), the single global threshold flagged 98% of Bangladeshi test respondents, and the secondary medical-borrowing model reached AUROC 0.71–0.73. With more time I would add country-specific calibration and thresholds through a multilevel model, economy-level inflation and social-protection covariates, and a wider boosting search (max_leaf_nodes settled on its upper bound), and, above all, move to data that link household finances to measured health (BDHS/STEPS biomarkers or a Dhaka household panel) to test the cardiometabolic and integrated-care parts of the research question.